# G5: chạy Target T0 (Hybrid + BGE Rerank) trên Test Set

Phần Người 1 Team 2, VLRA-78 (T2-39). B0 (BM25) đã chạy trên máy thường; T0 cần Dense và BGE nên chạy ở đây.

Chọn **Runtime → Change runtime type → GPU**, rồi chạy từ trên xuống. Ô 5 hỏi `GEMINI_API_KEY`
(chỉ giữ trong phiên, không ghi ra file). Mỗi câu gọi Gemini một lần, tốn quota.

Xong thì tải ZIP, giải nén `t0.json` vào `docs/team2_rag/runs/g5/t0.json` trong repo rồi chạy
`python scripts/benchmark_g5.py report --b0 docs/team2_rag/runs/g5/b0.json --t0 docs/team2_rag/runs/g5/t0.json`.

## 1. Lấy code và cài thư viện

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Chỉ dùng PyTorch; tránh framework khác cấp trước VRAM của Colab.
os.environ["HF_HOME"] = "/content/hf-cache"
os.environ["TORCH_HOME"] = "/content/torch-cache"
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

REPO_URL = "https://github.com/Banh-Van-Hiep/vietnamese-legal-rag.git"
BRANCH = "develop"  # cần code có scripts/benchmark_g5.py và RAG_RERANK_DEVICE
REPO = Path("/content/vietnamese-legal-rag")
if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
else:
    # Giữ checkout đã có; không pull hoặc xóa thay đổi khi chạy lại ô.
    origin = subprocess.check_output(["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True).strip()
    if origin != REPO_URL:
        raise RuntimeError("Thư mục hiện có không thuộc repo này; hãy dùng runtime mới.")
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
HEAD = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
if subprocess.check_output(["git", "status", "--porcelain"], text=True).strip():
    raise RuntimeError("Checkout có thay đổi chưa commit; hãy kiểm tra trước khi đo.")
print("Code:", HEAD)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[test,api,retrieval,dense,reranker]"], check=True)

## 2. Kiểm tra GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Chưa có CUDA GPU. Chọn GPU trong Runtime → Change runtime type, rồi kết nối lại.")
DEVICE = "cuda"
print("GPU:", torch.cuda.get_device_name())
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)

## 3. Test Set và cấu hình

Phải giống lượt B0: cùng Test Set, model Gemini và `RAG_TOP_K`. Báo cáo sẽ liệt kê mọi khác biệt.

In [ ]:
import hashlib
import json
from collections import Counter
from datetime import datetime, timezone
from huggingface_hub import model_info
from scripts.benchmark_g5 import load_cases

# Giống hệt lượt B0 chạy trên máy: cùng Test Set, cùng LLM, cùng top_k (xem docs/team2_rag/runs/g5/b0.json).
TEST_SET = REPO / "team1_data/golden/test_set_v1.json"
CHUNKS = REPO / "data/doanh-nghiep/processed/chunks.jsonl"
LLM_MODEL = "gemini-3.1-flash-lite"
TOP_K = "10"
EMBEDDING_MODEL = "AITeamVN/Vietnamese_Embedding"
EMBEDDING_REVISION = "dea33aa1ab339f38d66ae0a40e6c40e0a9249568"

cases = load_cases([TEST_SET], exploratory=False)  # từ chối nếu có câu không phải split=test
chunks = [json.loads(line) for line in CHUNKS.read_text(encoding="utf-8").splitlines() if line.strip()]
EMBEDDING_REVISION = model_info(EMBEDDING_MODEL, revision=EMBEDDING_REVISION).sha
CORPUS_HASH = hashlib.sha256(CHUNKS.read_bytes()).hexdigest()
VERSION = f"colab-{CORPUS_HASH[:12]}-{EMBEDDING_REVISION[:8]}"
RUN = Path("/content/legal-rag-runs") / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
EVIDENCE = RUN / "evidence"
EVIDENCE.mkdir(parents=True, exist_ok=False)
BM25_INDEX, DENSE_INDEX, QDRANT = [RUN / name for name in ("bm25", "dense", "qdrant")]
settings = dict(head=HEAD, corpus_sha256=CORPUS_HASH, index_version=VERSION, embedding_model=EMBEDDING_MODEL,
                embedding_revision=EMBEDDING_REVISION, device=DEVICE, gpu=torch.cuda.get_device_name(),
                torch=torch.__version__, cuda=torch.version.cuda)
(EVIDENCE / "run-settings.json").write_text(json.dumps(settings, ensure_ascii=False, indent=2), encoding="utf-8")
(EVIDENCE / "packages.txt").write_bytes(subprocess.check_output([sys.executable, "-m", "pip", "freeze"]))
print("Số câu:", len(cases), dict(Counter(c["expected_status"] for c in cases)))
print("Corpus:", len(chunks), "chunks; index:", VERSION)

## 4. Dựng index BM25 + Dense trên Colab

In [ ]:
import shutil

# Tiến trình build kết thúc sẽ trả toàn bộ VRAM trước khi runner nạp hai model.
BUILD_CODE = r"""
import json
import sys
from pathlib import Path

settings = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))
CHUNKS = Path(sys.argv[2])
EVIDENCE = Path(sys.argv[1]).parent
RUN = EVIDENCE.parent
BM25_INDEX, DENSE_INDEX, QDRANT = [RUN / name for name in ("bm25", "dense", "qdrant")]
VERSION = settings["index_version"]
EMBEDDING_MODEL, EMBEDDING_REVISION = settings["embedding_model"], settings["embedding_revision"]
DEVICE = settings["device"]
corpus = {json.loads(line)["chunk_id"] for line in CHUNKS.read_text(encoding="utf-8").splitlines() if line.strip()}

import shutil
import time
from qdrant_client import QdrantClient
from team1_data.retrieval import BM25Retriever, DenseRetriever, SentenceTransformerEmbedder

if any(path.exists() for path in (BM25_INDEX, DENSE_INDEX, QDRANT)):
    raise RuntimeError("Run này đã có index. Chạy lại từ ô 3 để tạo run mới.")
start = time.perf_counter()
sparse = BM25Retriever.from_jsonl(CHUNKS)
if set(sparse.chunk_ids) != set(corpus):
    raise ValueError("Bộ lọc hiệu lực làm thay đổi tập chunk; cần đối chiếu snapshot trước khi đo.")
sparse.save(BM25_INDEX, index_version=VERSION)
embedder = SentenceTransformerEmbedder(name=EMBEDDING_MODEL, revision=EMBEDDING_REVISION,
                                       batch_size=8, device=DEVICE)
client = QdrantClient(path=str(QDRANT))
try:
    dense = DenseRetriever.from_jsonl(CHUNKS, embedder=embedder, client=client, index_version=VERSION)
    if set(dense.chunk_ids) != set(sparse.chunk_ids):
        raise ValueError("Hai index không cùng tập chunk IDs.")
    dense.save(DENSE_INDEX)
finally:
    client.close()
for name, path in (("bm25", BM25_INDEX), ("dense", DENSE_INDEX)):
    shutil.copy2(path / "index_manifest.json", EVIDENCE / f"{name}-manifest.json")
(EVIDENCE / "build-time.json").write_text(json.dumps(dict(elapsed_ms=(time.perf_counter() - start) * 1000)))
print("Đã dựng index:", VERSION)
"""
subprocess.run([sys.executable, "-c", BUILD_CODE, str(EVIDENCE / "run-settings.json"), str(CHUNKS)],
               cwd=REPO, check=True)
print("Đã dựng index:", VERSION)

## 5. Chạy T0

BGE chạy trên GPU (`RAG_RERANK_DEVICE=cuda`) để không vượt hạn rerank 4 giây. Câu lỗi Gemini được thử lại
tối đa 2 lần, cùng quy tắc với B0; vẫn lỗi thì giữ trong kết quả.

In [ ]:
import getpass

# Key chỉ nằm trong biến môi trường của phiên Colab này: không in, không ghi file, không đóng gói.
if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass.getpass("GEMINI_API_KEY: ")
env = {**os.environ, "LLM_PROVIDER": "gemini", "LLM_MODEL": LLM_MODEL, "RAG_TOP_K": TOP_K,
       "RAG_BM25_INDEX": str(BM25_INDEX), "RAG_DENSE_INDEX": str(DENSE_INDEX), "RAG_QDRANT_PATH": str(QDRANT),
       "RAG_CHUNKS": str(CHUNKS), "RAG_RERANK_DEVICE": DEVICE}
OUT = EVIDENCE / "t0.json"
# Chạy lại ô này sau khi mất kết nối: runner bỏ qua câu đã có trong OUT.
result = subprocess.run([sys.executable, "scripts/benchmark_g5.py", "run", "--system", "t0",
                         "--dataset", str(TEST_SET), "--out", str(OUT), "--throttle", "2", "--retries", "2"],
                        cwd=REPO, env=env)
print("Exit code:", result.returncode)
if OUT.exists():
    t0 = json.loads(OUT.read_text(encoding="utf-8"))
    print("Đã xong:", len(t0["records"]), "/", len(cases), "câu;", dict(Counter(r["outcome"] for r in t0["records"].values())))
    print("Lùi về Hybrid do rerank lỗi/quá hạn:",
          sum(r["rerank_fallback_reason"] not in (None, "rerank_disabled") for r in t0["records"].values()))

## 6. Tải kết quả về máy

In [ ]:
import base64
from IPython.display import HTML, display

archive = shutil.make_archive(str(RUN / "g5-t0"), "zip", root_dir=EVIDENCE)
payload = base64.b64encode(Path(archive).read_bytes()).decode("ascii")
display(HTML(f'<a download="g5-t0.zip" href="data:application/zip;base64,{payload}">Tải ZIP kết quả T0</a>'))